# Solución Ejercicios - Administración de Bases de Datos

Este notebook contiene las soluciones a todos los ejercicios prácticos planteados en las diapositivas de la carpeta Administracion.

## 1. Transacciones (GD - 09 - Transacciones.pdf)

### Ejercicio: Tienda en línea
**Enunciado:** Paso a paso para una transacción completa: Buscar productos, Agregar al carrito, Confirmar orden, Validar inventario, Realizar pago, Validar pago, Actualizar inventario, Notificar usuario, Generar registro compra, Despachar, Acuse recibo, Finalizar pedido. ¿Dónde iniciaría una transacción y dónde terminaría?

**Solución (Discusión):**
La transacción a nivel de base de datos debería encapsular estrictamente las operaciones críticas de modificación de datos que deben ser atómicas para mantener la consistencia. Iniciaría justo antes de `Actualizar inventario` (o `Realizar pago/Validar pago` si esto implica modificar saldo en bd interna) y terminaría justo después de `Generar registro de compra`.
Las acciones como 'Buscar productos', 'Agregar al carrito', y 'Notificar a usuario' no necesitan ser parte de esta transacción atómica principal de la compra final.

### Ejercicio: Transferencia de dinero
**Enunciado:** Dibujar diagrama para transferir dinero e identificar transacciones.

**Solución:**
1. Leer saldo cuenta origen.
2. Verificar fondos suficientes.
3. **[START TRANSACTION]**
4. Restar monto de cuenta origen (`UPDATE`).
5. Sumar monto a cuenta destino (`UPDATE`).
6. Registrar movimiento en historial (`INSERT`).
7. **[COMMIT]** (o ROLLBACK si ocurre error).

### Ejercicios de Consola (1 al 4)
**Solución Práctica:**
Estos ejercicios demuestran el comportamiento de `COMMIT`, `ROLLBACK`, y el nivel de aislamiento. 

*   **Ejercicio 1:** Al no usar transacciones explícitas (auto_commit habilitado), la Sesión 2 ve los cambios inmediatamente tras el UPDATE en Sesión 1.
*   **Ejercicio 2:** Al usar `START TRANSACTION` en S1, S2 no ve los cambios hasta que S1 ejecuta `COMMIT` (asumiendo aislamiento READ COMMITTED o REPEATABLE READ).
*   **Ejercicio 3:** Similar al anterior, pero al ejecutar `ROLLBACK` en S1, los cambios se deshacen y S2 nunca los ve aplicados de forma definitiva.
*   **Ejercicio 4:** Muestra el uso de `SAVEPOINT`. S1 hace varios UPDATES con savepoints entre ellos. Al hacer `ROLLBACK TO savepoint`, deshace solo parte del trabajo, y luego un `COMMIT` hace definitivos los cambios que quedaron.

## 2. Concurrencia (GD - 10 - Concurrencia.pdf)

### Ejercicio: Abrazo Mortal (Deadlock)
**Enunciado:** Escribir código sql en MariaDB que dé como resultado un abrazo mortal.

In [ ]:
-- SESIÓN 1
START TRANSACTION;
UPDATE world.countries SET population = population + 1 WHERE code = 'COL';
-- (Esperar a que Sesión 2 haga su primer UPDATE)
UPDATE world.countries SET population = population + 1 WHERE code = 'ARG';

-- SESIÓN 2
START TRANSACTION;
UPDATE world.countries SET population = population + 1 WHERE code = 'ARG';
-- (Esperar a que Sesión 1 haga su primer UPDATE)
UPDATE world.countries SET population = population + 1 WHERE code = 'COL';

-- Resultado: Una de las dos sesiones mostrará el error "Deadlock found when trying to get lock; try restarting transaction".

### Ejercicio: Nivel de Aislamiento
**Enunciado:** Investigar cómo configurar y cambiar el nivel de aislamiento en MariaDB.

In [ ]:
-- Consultar el nivel de aislamiento actual
SELECT @@GLOBAL.tx_isolation, @@tx_isolation;
-- o en versiones recientes:
SELECT @@GLOBAL.transaction_isolation, @@SESSION.transaction_isolation;

-- Cambiar nivel a READ UNCOMMITTED para la sesión actual
SET SESSION TRANSACTION ISOLATION LEVEL READ UNCOMMITTED;

-- Cambiar nivel a SERIALIZABLE globalmente
SET GLOBAL TRANSACTION ISOLATION LEVEL SERIALIZABLE;

## 3. Usuarios y Roles (GD - 11 - Usuarios y roles.pdf)

### Ejercicio 1 y 2: Creación de Roles y Usuarios
**Enunciado:** Crear roles developer, qa, business. Configurar usuarios para el equipo dado con nombre de usuario: primera letra nombre + apellidos (max 8 chars). Configurar IPs y límites.

In [ ]:
-- 1. CREACIÓN DE ROLES
CREATE ROLE developer;
CREATE ROLE qa;
CREATE ROLE business;

-- 2. CREACIÓN DE USUARIOS (Asumiendo contraseñas genéricas para el ejemplo)
-- Desarrolladores (Sin restricciones, cualquier IP '%')
CREATE USER 'ldavinci'@'%' IDENTIFIED BY 'pass123';
CREATE USER 'inewton'@'%' IDENTIFIED BY 'pass123';
CREATE USER 'ntesla'@'%' IDENTIFIED BY 'pass123';
CREATE USER 'mfaraday'@'%' IDENTIFIED BY 'pass123';

-- QA (Desde subred actual, digamos 192.168.5.0/255.255.255.0. Límites: 30 consultas, 40 updates, 1 conexión)
CREATE USER 'ggalilei'@'192.168.5.0/255.255.255.0' IDENTIFIED BY 'pass123'
  WITH MAX_QUERIES_PER_HOUR 30 MAX_UPDATES_PER_HOUR 40 MAX_USER_CONNECTIONS 1;
CREATE USER 'mcurie'@'192.168.5.0/255.255.255.0' IDENTIFIED BY 'pass123'
  WITH MAX_QUERIES_PER_HOUR 30 MAX_UPDATES_PER_HOUR 40 MAX_USER_CONNECTIONS 1;
CREATE USER 'cdarwin'@'192.168.5.0/255.255.255.0' IDENTIFIED BY 'pass123'
  WITH MAX_QUERIES_PER_HOUR 30 MAX_UPDATES_PER_HOUR 40 MAX_USER_CONNECTIONS 1;

-- Business (IPs específicas. Límite: 1 conexión)
CREATE USER 'gbell'@'192.168.5.51' IDENTIFIED BY 'pass123'
  WITH MAX_USER_CONNECTIONS 1;
CREATE USER 'ncoperni'@'192.168.5.52' IDENTIFIED BY 'pass123'
  WITH MAX_USER_CONNECTIONS 1;

-- El director (Albert Einstein) no ingresa, por lo que no se crea usuario.

-- 3. ASIGNACIÓN DE ROLES A USUARIOS
GRANT developer TO 'ldavinci'@'%', 'inewton'@'%', 'ntesla'@'%', 'mfaraday'@'%';
GRANT qa TO 'ggalilei'@'192.168.5.0/255.255.255.0', 'mcurie'@'192.168.5.0/255.255.255.0', 'cdarwin'@'192.168.5.0/255.255.255.0';
GRANT business TO 'gbell'@'192.168.5.51', 'ncoperni'@'192.168.5.52';


## 4. Permisos y Privilegios (GD - 12 - Permisos y privilegios.pdf)
### Ejercicio 1: Asignación de Permisos a Roles
**Enunciado:** Usando los roles de la clase anterior, definir permisos:
*   **developer:** insert, select en cities y countries. crear, modificar, eliminar rutinas.
*   **qa:** insert, select, update, delete en cities, countries, languages, countries_languages.
*   **business:** select sobre instancia completa.

In [ ]:
-- Permisos para el rol 'developer'
GRANT INSERT, SELECT ON world.cities TO developer;
GRANT INSERT, SELECT ON world.countries TO developer;
GRANT CREATE ROUTINE, ALTER ROUTINE ON world.* TO developer; -- Depende del motor, en MariaDB es sobre BD

-- Permisos para el rol 'qa'
GRANT INSERT, SELECT, UPDATE, DELETE ON world.cities TO qa;
GRANT INSERT, SELECT, UPDATE, DELETE ON world.countries TO qa;
GRANT INSERT, SELECT, UPDATE, DELETE ON world.languages TO qa;
GRANT INSERT, SELECT, UPDATE, DELETE ON world.countries_languages TO qa;

-- Permisos para el rol 'business'
GRANT SELECT ON *.* TO business;

FLUSH PRIVILEGES;

## 5. Vistas (GD - 13 - Vistas.pdf)
### Ejercicio 1
**Enunciado:** Crear vista 'v_countries_01' (id, name, relative_size basado en área).

In [ ]:
CREATE OR REPLACE VIEW v_countries_01 AS
SELECT 
    code AS id, 
    name,
    CASE 
        WHEN area <= 1000000 THEN 'SMALL'
        WHEN area <= 5000000 THEN 'MEDIUM'
        ELSE 'BIG'
    END AS relative_size
FROM countries;

### Ejercicio 2
**Enunciado:** Crear 'v_countries_02' igual a anterior pero solo norte america y sur america.

In [ ]:
CREATE OR REPLACE VIEW v_countries_02 AS
SELECT 
    code AS id, 
    name,
    CASE 
        WHEN area <= 1000000 THEN 'SMALL'
        WHEN area <= 5000000 THEN 'MEDIUM'
        ELSE 'BIG'
    END AS relative_size
FROM countries
WHERE continent IN ('North America', 'South America');

### Ejercicio 3
**Enunciado:** Crear vista con code, name, continent, independence_year y campo calculado de hace cuántos años obtuvo independencia.

In [ ]:
CREATE OR REPLACE VIEW v_countries_03 AS
SELECT 
    code,
    name,
    continent,
    independence_year,
    IF(independence_year IS NOT NULL, YEAR(CURDATE()) - independence_year, NULL) AS years_since_independence
FROM countries;

### Ejercicio 4
**Enunciado:** Crear 'v_countries_04' con id, language_name, is_official, mostrando idiomas en todos los países.

In [ ]:
CREATE OR REPLACE VIEW v_countries_04 AS
SELECT 
    c.code AS id,
    cl.language_id AS language_name,
    cl.is_official
FROM countries c
JOIN countries_languages cl ON c.code = cl.country_id;

### Ejercicio 5
**Enunciado:** Crear 'v_cities_countries_01' con id pais, id ciudad, nombre pais, nombre ciudad. Ordenado alfabeticamente ascendente primero por pais y luego ciudad.

In [ ]:
CREATE OR REPLACE VIEW v_cities_countries_01 AS
SELECT 
    co.code AS country_id,
    ci.id AS city_id,
    co.name AS country_name,
    ci.name AS city_name
FROM countries co
JOIN cities ci ON co.code = ci.country_id
ORDER BY co.name ASC, ci.name ASC;